In [1]:
import pandas as pd

data = pd.read_csv("./data/nagasaki.csv")

print(data)

area = data["選挙区名"].to_list()
peopleNum = data["当日有権者数"].to_numpy()
print(area)
print(peopleNum)

         選挙区名  当日有権者数
0         長崎市  337717
1   佐世保市・北松浦郡  198187
2         島原市   35826
3         諫早市  110131
4         大村市   77597
5         平戸市   24792
6         松浦市   18296
7         対馬市   23892
8         壱岐市   21299
9         五島市   28683
10        西海市   22424
11        雲仙市   34705
12       南島原市   35463
13       西彼杵郡   59203
14       東彼杵郡   29833
15       南松浦郡   13427
['長崎市', '佐世保市・北松浦郡', '島原市', '諫早市', '大村市', '平戸市', '松浦市', '対馬市', '壱岐市', '五島市', '西海市', '雲仙市', '南島原市', '西彼杵郡', '東彼杵郡', '南松浦郡']
[337717 198187  35826 110131  77597  24792  18296  23892  21299  28683
  22424  34705  35463  59203  29833  13427]


In [2]:
import numpy as np

# 2分ヒープ関連の関数
# https://ja.wikipedia.org/wiki/%E4%BA%8Cf%E5%88%86%E3%83%92%E3%83%BC%E3%83%97を参考にする。
# 最小値を求める。

class Heap:

    def __init__(self, func):
        self.func = func
        self.heapList = []

    def getParentIndex(self, i):
        return (int)(np.floor((i + 1)/2) - 1)

    def getChildIndice(self, i):
        if len(self.heapList) > 2*(i+1):
            retValue = [2*(i+1) - 1, 2*(i+1)]
        elif len(self.heapList) == 2*(i+1):
            retValue = [2*(i+1) - 1]
        else:
            retValue = []

        return retValue

    # データを追加する。
    # ループは木の深さになるのでlog2(n)回行う。
    def upHeap(self, data):
        self.heapList.append(data)

        lastIndex = len(self.heapList) - 1

        while lastIndex != 0:
            parentIndex = self.getParentIndex(lastIndex)
            if self.func(self.heapList[lastIndex][0], self.heapList[parentIndex][0]):
                tmp = self.heapList[lastIndex]
                self.heapList[lastIndex] = self.heapList[parentIndex]
                self.heapList[parentIndex] = tmp

                lastIndex = parentIndex
            else:
                break

    def addData(self, data):
        self.upHeap(data)

    # 根のデータを削除する
    # whileループは木の深さになるのでlog2(n)回行う。
    def downHeap(self):
        if len(self.heapList) > 0:
            # 最後のデータを最初のデータにする。
            self.heapList[0] = self.heapList[len(self.heapList) - 1]
            self.heapList = self.heapList[:-1]

            i = 0
            while True:
                childIndice = self.getChildIndice(i)
                # 子でより条件を満たすほうを見つける
                if len(childIndice) == 2:
                    if self.func(self.heapList[childIndice[0]][0], self.heapList[childIndice[1]][0]):
                        childIndex = childIndice[0]
                    else:
                        childIndex = childIndice[1]

                elif len(childIndice) == 1:
                    childIndex = childIndice[0]
                else:
                    break

                if self.func(self.heapList[childIndex][0], self.heapList[i][0]):
                    tmp = self.heapList[childIndex]
                    self.heapList[childIndex] = self.heapList[i]
                    self.heapList[i] = tmp
                    i = childIndex
                else:
                    break

    def removeRoot(self):
        self.downHeap()

    def getList(self):
        return self.heapList
    
    def getRootData(self):
        return self.heapList[0]

def minFunc(a, b):
    return (a < b)

testHeap = Heap(minFunc)

testHeap.addData([10, []])
print(testHeap.getList())
testHeap.addData([5, []])
print(testHeap.getList())
testHeap.addData([15, []])
print(testHeap.getList())
testHeap.addData([3, []])
print(testHeap.getList())
testHeap.addData([1, []])
print(testHeap.getList())
testHeap.addData([2, []])
print(testHeap.getList())
testHeap.addData([4, []])
print(testHeap.getList())
testHeap.removeRoot()
print(testHeap.getList())
testHeap.removeRoot()
print(testHeap.getList())
testHeap.removeRoot()
print(testHeap.getList())
testHeap.removeRoot()
print(testHeap.getList())
testHeap.removeRoot()
print(testHeap.getList())



[[10, []]]
[[5, []], [10, []]]
[[5, []], [10, []], [15, []]]
[[3, []], [5, []], [15, []], [10, []]]
[[1, []], [3, []], [15, []], [10, []], [5, []]]
[[1, []], [3, []], [2, []], [10, []], [5, []], [15, []]]
[[1, []], [3, []], [2, []], [10, []], [5, []], [15, []], [4, []]]
[[2, []], [3, []], [4, []], [10, []], [5, []], [15, []]]
[[3, []], [5, []], [4, []], [10, []], [15, []]]
[[4, []], [5, []], [15, []], [10, []]]
[[5, []], [10, []], [15, []]]
[[10, []], [15, []]]


In [3]:
import numpy as np
# 上のデータで(4.60)をやってみる。
# まずはヒープを使わずアルゴリズム4.1をやってみる。

#議員定数
B = 46

# step1 初期化
memberNum = np.zeros(peopleNum.shape[0])

# step2  値の確認
while np.sum(memberNum) < B: # この部分がオーダB
    # step3
    # 直接的に(4.60)の下の式を評価する。

    # dを求める。
    ds = [(2*x + 1) / p for x, p in zip(memberNum, peopleNum)] #　この部分がオーダn

    minIndex = ds.index(np.min(ds)) #　この部分がオーダn
    memberNum[minIndex] += 1

for x, a in zip(memberNum, area):
    print(a, x)

長崎市 14.0
佐世保市・北松浦郡 8.0
島原市 2.0
諫早市 5.0
大村市 3.0
平戸市 1.0
松浦市 1.0
対馬市 1.0
壱岐市 1.0
五島市 1.0
西海市 1.0
雲仙市 1.0
南島原市 2.0
西彼杵郡 3.0
東彼杵郡 1.0
南松浦郡 1.0


In [4]:
# ヒープを使ってみる。

#議員定数
B = 46

def minFunc(a, b):
    return (a < b)

# step1 初期化
memberNum = np.zeros(peopleNum.shape[0])

heap = Heap(minFunc)
for i, p in enumerate(peopleNum):
    # この操作がlog2(n)のオーダ
    heap.addData([1/p, [i]])

# step2  値の確認
while np.sum(memberNum) < B:
    # step3
    # 直接的に(4.60)の下の式を評価する。

    # 値の最大を求めるが、それはheapの根になっている。
    # この操作はオーダ1
    maxData = heap.getRootData()
    memberNum[maxData[1][0]] += 1

    # 値を更新するがheapの根のところだけ更新する。
    # 少し計算量は増えるが、根を消して、データを追加する作業とする。
    # 計算量はlog2(n)のオーダ
    heap.removeRoot()
    heap.addData([(2*memberNum[maxData[1][0]] + 1)/peopleNum[maxData[1][0]], [maxData[1][0]]])

for x, a in zip(memberNum, area):
    print(a, x)

長崎市 14.0
佐世保市・北松浦郡 8.0
島原市 2.0
諫早市 5.0
大村市 3.0
平戸市 1.0
松浦市 1.0
対馬市 1.0
壱岐市 1.0
五島市 1.0
西海市 1.0
雲仙市 1.0
南島原市 2.0
西彼杵郡 3.0
東彼杵郡 1.0
南松浦郡 1.0
